# 🛒 E-Commerce DA Project — Data Generator

이 노트북은 DA 프로젝트용 **가상 이커머스 데이터**를 생성합니다.

### 생성되는 파일
| 파일 | 내용 |
|------|------|
| `users.csv` | 유저 기본 정보 (50,000명) — 한 번만 생성 |
| `orders_YYYY-MM-DD.csv` | 일별 주문 데이터 (5,000~12,000건/일) |

### 셀 실행 순서
**1 → 2 → 3 → 4** 순서대로 실행하세요. 각 셀 왼쪽 ▶ 버튼을 누르면 됩니다.

---
## Cell 1 — 라이브러리 불러오기
Colab에는 pandas, numpy가 기본 설치돼 있어서 따로 설치 안 해도 됩니다.

In [ ]:
import pandas as pd
import numpy as np
import random
import os
from datetime import datetime, timedelta

print('✅ Libraries loaded successfully!')

---
## Cell 2 — 설정값 & 데이터 정의
여기서 하루 주문 건수, 유저 수 등을 바꿀 수 있어요.
처음엔 그냥 그대로 실행하세요.

In [ ]:
# ──────────────────────────────────────────
# 설정값 (바꾸고 싶으면 여기만 수정하세요)
# ──────────────────────────────────────────
DAILY_MIN   = 5000    # 하루 최소 주문 수
DAILY_MAX   = 12000   # 하루 최대 주문 수
TOTAL_USERS = 50000   # 전체 유저 수
OUTPUT_DIR  = '/content/ecommerce_data'  # Colab 저장 폴더
# ──────────────────────────────────────────


# 상품 카테고리 & 가격 범위 (원)
CATEGORIES = {
    'Electronics':    (50000,  1500000),
    'Fashion':        (10000,  300000),
    'Beauty':         (5000,   150000),
    'Food':           (3000,   80000),
    'Sports':         (15000,  500000),
    'Home & Living':  (8000,   400000),
    'Books':          (8000,   50000),
    'Baby & Toys':    (10000,  200000),
}

# 광고 채널 & 비율
AD_CHANNELS = ['Search Ad', 'SNS Ad', 'Display Ad', 'Email', 'Organic']
AD_WEIGHTS  = [0.20, 0.25, 0.15, 0.10, 0.30]

# 지역
REGIONS = [
    'Seoul', 'Gyeonggi', 'Incheon', 'Busan', 'Daegu',
    'Daejeon', 'Gwangju', 'Ulsan', 'Gangwon', 'Jeju'
]

# 결제 수단
PAYMENTS = ['Credit Card', 'Debit Card', 'Kakao Pay', 'Naver Pay', 'Bank Transfer']

# 디바이스
DEVICES         = ['Mobile', 'PC', 'Tablet']
DEVICE_WEIGHTS  = [0.65, 0.30, 0.05]

# 주문 상태
ORDER_STATUSES  = ['Paid', 'Shipping', 'Delivered', 'Cancelled', 'Returned']
STATUS_WEIGHTS  = [0.20, 0.30, 0.40, 0.07, 0.03]

print('✅ Config ready!')
print(f'   Users: {TOTAL_USERS:,} | Daily orders: {DAILY_MIN:,}~{DAILY_MAX:,}')
print(f'   Output folder: {OUTPUT_DIR}')

---
## Cell 3 — 데이터 생성 함수 정의
실행해도 아직 파일이 만들어지지 않아요. 함수를 '등록'하는 단계입니다.

In [ ]:
def make_users():
    """유저 테이블 생성 — 한 번만 만들면 됩니다"""
    np.random.seed(42)  # 항상 같은 유저가 나오도록 고정

    n = TOTAL_USERS
    ages = np.random.randint(18, 65, size=n)

    # 나이대 구분
    def age_group(age):
        if age < 25: return '18-24'
        elif age < 35: return '25-34'
        elif age < 45: return '35-44'
        elif age < 55: return '45-54'
        else: return '55+'

    users = pd.DataFrame({
        'user_id':            [f'U{str(i).zfill(6)}' for i in range(1, n+1)],
        'gender':             np.random.choice(['Male', 'Female'], size=n, p=[0.48, 0.52]),
        'age':                ages,
        'age_group':          [age_group(a) for a in ages],
        'region':             np.random.choice(REGIONS, size=n),
        'signup_date':        [
                                  (datetime(2023,1,1) + timedelta(days=random.randint(0,729))).strftime('%Y-%m-%d')
                                  for _ in range(n)
                              ],
        'user_grade':         np.random.choice(['Bronze','Silver','Gold','VIP'], size=n, p=[0.50,0.25,0.15,0.10]),
        'ad_consent':         np.random.choice([True, False], size=n, p=[0.70, 0.30]),
        'preferred_category': np.random.choice(list(CATEGORIES.keys()), size=n),
        'past_purchase_count':np.random.randint(0, 50, size=n),
    })
    return users


def make_orders(date_str, users_df):
    """하루치 주문 테이블 생성"""
    n = random.randint(DAILY_MIN, DAILY_MAX)

    # 유저 랜덤 샘플링 (한 유저가 여러 번 구매 가능)
    sampled = users_df.sample(n=n, replace=True).reset_index(drop=True)

    # 카테고리 & 가격
    categories = np.random.choice(list(CATEGORIES.keys()), size=n)
    unit_prices = [
        round(random.randint(CATEGORIES[c][0]//1000, CATEGORIES[c][1]//1000) * 1000)
        for c in categories
    ]

    # 광고 채널
    ad_channels = np.random.choice(AD_CHANNELS, size=n, p=AD_WEIGHTS)
    ad_exposed  = (ad_channels != 'Organic').astype(int)  # 1=광고유입, 0=자연유입

    # 광고 유입 유저는 구매금액 약간 높게 (광고 효과 시뮬레이션)
    unit_prices = [
        int(p * random.uniform(1.0, 1.15)) if exp else p
        for p, exp in zip(unit_prices, ad_exposed)
    ]

    # 수량
    quantities = np.random.choice([1,2,3,4,5], size=n, p=[0.70,0.15,0.08,0.04,0.03])

    # 할인
    has_discount  = np.random.choice([True, False], size=n, p=[0.35, 0.65])
    discount_rate = [
        round(random.choice([0.05, 0.10, 0.15, 0.20, 0.30]), 2) if d else 0.0
        for d in has_discount
    ]
    final_amount = [
        int(p * q * (1 - dr))
        for p, q, dr in zip(unit_prices, quantities, discount_rate)
    ]

    # 주문 시간 (점심/저녁대 많음)
    hour_w = [1,1,1,1,1,2, 3,4,5,6,7,8, 9,9,8,7,8,10, 12,10,8,6,4,2]
    hours   = np.random.choice(range(24), size=n, p=np.array(hour_w)/sum(hour_w))
    minutes = np.random.randint(0, 60, size=n)
    order_times = [f"{date_str} {str(h).zfill(2)}:{str(m).zfill(2)}:00" for h,m in zip(hours,minutes)]

    orders = pd.DataFrame({
        # 주문 정보
        'order_id':       [f"ORD-{date_str.replace('-','')}-{str(i).zfill(6)}" for i in range(1, n+1)],
        'order_datetime': order_times,
        'order_date':     date_str,

        # 유저 정보 (분석 편하게 주문 테이블에 포함)
        'user_id':        sampled['user_id'].values,
        'gender':         sampled['gender'].values,
        'age_group':      sampled['age_group'].values,
        'region':         sampled['region'].values,
        'user_grade':     sampled['user_grade'].values,

        # 상품 정보
        'category':       categories,
        'unit_price':     unit_prices,
        'quantity':       quantities,
        'discount_rate':  discount_rate,
        'final_amount':   final_amount,

        # 광고 관련 (핵심!)
        'ad_channel':     ad_channels,
        'ad_exposed':     ad_exposed,   # 1=광고유입, 0=자연유입

        # 기타
        'device':         np.random.choice(DEVICES, size=n, p=DEVICE_WEIGHTS),
        'payment_method': np.random.choice(PAYMENTS, size=n),
        'order_status':   np.random.choice(ORDER_STATUSES, size=n, p=STATUS_WEIGHTS),

        # AB 테스트용
        'ab_test_group':  np.random.choice(['A','B'], size=n),
    })
    return orders


print('✅ Functions defined!')

---
## Cell 4 — 데이터 생성 실행

아래 두 가지 중 **하나만** 실행하세요.

- **4-A**: 오늘 하루치만 빠르게 테스트
- **4-B**: 특정 기간 전체 생성 (예: 2025-01-01 ~ 오늘)

In [ ]:
# ── Cell 4-A : 오늘 하루치만 생성 (테스트용) ──────────────────

os.makedirs(f'{OUTPUT_DIR}/users',  exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/orders', exist_ok=True)

# 유저 테이블 생성 (파일 없을 때만)
user_path = f'{OUTPUT_DIR}/users/users.csv'
if not os.path.exists(user_path):
    print('Creating user table...')
    users_df = make_users()
    users_df.to_csv(user_path, index=False)
    print(f'  → Saved {len(users_df):,} users')
else:
    users_df = pd.read_csv(user_path)
    print(f'Loaded existing users: {len(users_df):,}')

# 오늘 주문 생성
today = datetime.today().strftime('%Y-%m-%d')
orders_df = make_orders(today, users_df)
orders_df.to_csv(f'{OUTPUT_DIR}/orders/orders_{today}.csv', index=False)

print(f'\n✅ Done! orders_{today}.csv')
print(f'   Orders   : {len(orders_df):,} rows')
print(f'   Revenue  : ₩{orders_df["final_amount"].sum():,}')
print(f'   Ad rate  : {orders_df["ad_exposed"].mean()*100:.1f}%')
orders_df.head(3)

In [ ]:
# ── Cell 4-B : 기간 전체 생성 ──────────────────────────────────
# 날짜 범위를 바꾸고 실행하세요 (시간이 좀 걸려요)

START_DATE = '2025-01-01'
END_DATE   = datetime.today().strftime('%Y-%m-%d')  # 오늘까지

os.makedirs(f'{OUTPUT_DIR}/users',  exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/orders', exist_ok=True)

# 유저 테이블
user_path = f'{OUTPUT_DIR}/users/users.csv'
if not os.path.exists(user_path):
    print('Creating user table...')
    users_df = make_users()
    users_df.to_csv(user_path, index=False)
    print(f'  → Saved {len(users_df):,} users')
else:
    users_df = pd.read_csv(user_path)
    print(f'Loaded existing users: {len(users_df):,}')

# 날짜 루프
current = datetime.strptime(START_DATE, '%Y-%m-%d')
end     = datetime.strptime(END_DATE,   '%Y-%m-%d')
all_orders = []

total_days = (end - current).days + 1
print(f'\nGenerating {total_days} days of data ({START_DATE} ~ {END_DATE})...')

while current <= end:
    date_str  = current.strftime('%Y-%m-%d')
    df        = make_orders(date_str, users_df)
    df.to_csv(f'{OUTPUT_DIR}/orders/orders_{date_str}.csv', index=False)
    all_orders.append(df)
    current  += timedelta(days=1)

# 전체 합본 저장 (Snowflake 업로드용)
combined = pd.concat(all_orders, ignore_index=True)
combined.to_csv(f'{OUTPUT_DIR}/orders_all.csv', index=False)

print(f'\n✅ All done!')
print(f'   Total orders : {len(combined):,} rows')
print(f'   Total revenue: ₩{combined["final_amount"].sum():,}')
print(f'   Saved to     : {OUTPUT_DIR}/orders_all.csv')
combined.head(3)

---
## Cell 5 — 간단 확인 (데이터 미리 보기)
생성된 데이터가 어떻게 생겼는지 확인해보세요.

In [ ]:
# 주문 데이터 기본 통계
print('=== Order Data Overview ===')
print(f'Shape       : {orders_df.shape}')
print(f'Date range  : {orders_df["order_date"].min()} ~ {orders_df["order_date"].max()}')
print()

print('--- Ad Channel Distribution ---')
print(orders_df['ad_channel'].value_counts(normalize=True).map('{:.1%}'.format))
print()

print('--- Avg Order Amount by Ad Exposure ---')
print(orders_df.groupby('ad_exposed')['final_amount'].mean().map('₩{:,.0f}'.format))
print()

print('--- Category Distribution ---')
print(orders_df['category'].value_counts())

---
## Cell 6 — 파일 다운로드 (Colab → 내 컴퓨터)
생성된 CSV 파일을 로컬로 다운받을 수 있어요.

In [ ]:
from google.colab import files

# 합본 파일 다운로드
files.download(f'{OUTPUT_DIR}/orders_all.csv')
files.download(f'{OUTPUT_DIR}/users/users.csv')

print('✅ Download started!')